# Act 1 — The Cloud Load Balancing family

GCP's Cloud Load Balancing is one product line with many shapes. The shapes are organised on three axes:

- **Where it's reachable from** — external (public internet) or internal (inside the VPC).
- **What layer it operates at** — Application (L7, HTTPS) or Network (L4, TCP/UDP).
- **Geographic scope** — global (one anycast IP everywhere) or regional (one region).

Not every combination exists, but most do. The matrix below is the cheat sheet to come back to whenever someone asks "which LB?"

## The LB matrix

| LB type | Reachable | Layer | Scope | Use when |
|---|---|---|---|---|
| **Global External Application LB** | Public | L7 (HTTPS) | Global anycast | Headline LB for any public HTTP/HTTPS service |
| **Regional External Application LB** | Public | L7 (HTTPS) | One region | Regional-only public HTTPS with simpler ops |
| **Internal Application LB** | Private (inside VPC) | L7 (HTTPS) | Regional (or cross-region) | Internal microservice routing |
| **External Network LB (passthrough)** | Public | L4 (TCP/UDP) | Regional | Non-HTTP protocols, gaming, sub-millisecond paths |
| **Internal Network LB (passthrough)** | Private | L4 (TCP/UDP) | Regional | Internal database/cache front, NLB-style internal traffic |
| **External Network LB (proxy)** | Public | L4 (TCP/SSL) | Global or regional | Long-lived TCP with TLS termination |
| **Internal Network LB (proxy)** | Private | L4 (TCP/SSL) | Regional | Internal long-lived TCP |

**The headline shape — the one this chapter focuses on — is the Global External Application LB.** It's the one Cloud Run, GKE Ingress, and most public HTTPS workloads sit behind. It terminates TLS at the Google Front End, evaluates Cloud Armor, hits Cloud CDN, then forwards to your backends over Google's private network.

**Two network service tiers** affect external LBs and external IPs in general:

- **Premium tier** — traffic ingresses at the nearest Google PoP and rides Google's backbone the entire way. Default for external HTTPS LB.
- **Standard tier** — traffic stays on the public internet until the destination region. Cheaper, lower performance, less consistent. Acceptable for low-traffic or regional-only workloads.

# Act 2 — Anatomy of a Global External Application LB

The Global External Application LB is a chain of resources, not one resource. Walking through the chain end-to-end is the fastest way to understand what each piece does, because every other LB variant uses some subset of the same chain.

## The chain — five pieces in order

```
Forwarding Rule → Target HTTPS Proxy → URL Map → Backend Service → Backend (MIG / NEG)
```

1. **Forwarding Rule** — binds an external anycast IP + port (e.g. `35.241.x.x:443`) to a target proxy. This is the customer-facing resource that the public DNS record points at.
2. **Target HTTPS Proxy** — terminates TLS (you attach a managed or self-managed SSL certificate). For HTTP-only, it's a Target HTTP Proxy with no cert.
3. **URL Map** — the L7 routing rules. Match on host, path, header, or query; dispatch to different backend services. `/api/*` → API backend, `/static/*` → CDN-fronted GCS backend, everything else → web backend.
4. **Backend Service** — the load-balancing policy plus the list of backends. Health checks, session affinity, balancing mode (RATE / UTILIZATION / CONNECTION), CDN attachment, Cloud Armor attachment.
5. **Backend** — what actually serves the request. Three shapes:
   - **Instance Group** (MIG or unmanaged) — Compute Engine VMs.
   - **Network Endpoint Group (NEG)** — Cloud Run, App Engine, Cloud Functions, or arbitrary IP:port pairs.
   - **Serverless NEG** — the modern way to put Cloud Run behind a Global External Application LB.

The chain looks bureaucratic the first time. Once internalised, it's flexible: one URL map can fan out to multiple backend services; one backend service can serve multiple URL maps; backends can mix MIGs and serverless NEGs in the same group.

**Health checks** live on the backend service. The LB only sends traffic to instances that pass the health check. Recap from notebook 03: LB health checks are configured separately from MIG autohealing health checks; they often look identical (`/healthz` returning 200) but live in different resources.

## SSL certificates

Two flavours of certificate for the Target HTTPS Proxy:

- **Google-managed** — Google provisions and renews a certificate for your domain via Let's Encrypt-style ACME. You point DNS at the LB's anycast IP, GCP completes the challenge, and the cert lives forever (auto-renewed). The right default for public HTTPS.
- **Self-managed** — upload your own cert and private key. Used when you have an existing CA contract, EV certs, or wildcards from outside Google.

Certificate Manager (the newer, separate service) handles SAN sprawl and DNS-authorized certs at scale — preferred over the older per-proxy SSL certificate resource for any deployment with more than ~10 domains.

# Act 3 — Edge protection: Cloud Armor and Cloud CDN

The Global External Application LB unlocks two edge-only features that are GCP's strongest network differentiators: a real WAF that runs at the Google Front End, and a CDN that's a one-checkbox attach to your existing backend service.

## Cloud Armor — WAF at the GFE

**Cloud Armor** is GCP's WAF. It runs at the Google Front End, *before* traffic reaches your backends, anywhere in the world. You attach an Armor policy to a backend service; the policy is a list of rules with priorities.

Three kinds of rule:

- **Custom CEL rules** — `inIpRange(origin.ip, '203.0.113.0/24')` to allow corporate IPs, `request.headers['user-agent'].contains('badbot')` to block by header.
- **Preconfigured WAF rules** — OWASP Top 10 (`sqli-stable`, `xss-stable`, `lfi-stable`, …). Switchable in detection-only mode first, then enforcement.
- **Rate-limiting rules** — `enforce_on_key: IP, rate_limit: 100/min, exceed_action: deny(429)`. Throttle abusive clients without dropping legitimate ones.

**Two policy types:**

- **Security policy** — attached to the backend service, applied per request at the GFE.
- **Edge security policy** — applied even earlier, before caching decisions; used for Cloud CDN-fronted content.

**Adaptive Protection** is the Armor Premium feature that uses ML to detect L7 DDoS patterns in your traffic and propose protective rules. Worth turning on for any service that's been the target of bot activity.

Notebook 11 covers Armor's positioning in the broader security stack alongside reCAPTCHA Enterprise.

## Cloud CDN — one-checkbox cache at the GFE

**Cloud CDN** caches HTTP responses at GFE edge locations. Enable it by ticking the `enable_cdn` field on a backend service — there's no separate resource to provision.

**Cache modes:**

- **Cache static content** — cache responses for `200/203/204/206/300/301/302/304/307/308/404/405/410/451/501` that carry Cache-Control headers indicating cacheability.
- **Cache all static content** — cache common static MIME types (images, fonts, CSS, JS) regardless of headers.
- **Force cache all content** — cache everything, including responses without cache headers. Use carefully.
- **Use origin headers** — fully driven by upstream `Cache-Control`. Default for serving from MIGs.

**Signed URLs and signed cookies** authenticate access to cached content — generate a signed URL on the application side, hand it to the user, the GFE checks the signature and serves from cache. Standard pattern for private CDN content.

**Negative caching** caches 404/410 responses for short periods so a deluge of "file not found" requests doesn't slam your origin.

# Act 4 — Names: Cloud DNS and Service Directory

The last piece of traffic flow is name resolution. Cloud DNS handles authoritative DNS for public and private zones; Service Directory is the discovery layer for service-to-service traffic inside the VPC.

## Cloud DNS

**Cloud DNS** is GCP's managed authoritative DNS. Anycast nameservers around the world; high reliability; programmatic management via API/Terraform.

**Two zone types:**

- **Public zone** — authoritative for a public domain (`acme.com`). Replaces (or sits alongside) your existing registrar's DNS.
- **Private zone** — visible only inside specified VPCs. Resolves `db.internal.acme.com` to a private IP for workloads inside the VPC.

**Features worth knowing:**

- **DNSSEC** — signed records for public zones. One-click enable; standard.
- **DNS peering** — make zones in one VPC resolvable from another VPC (typical with Shared VPC + service projects, or hub-and-spoke).
- **Forwarding zones** — for a given suffix, forward queries to specified resolvers (on-prem DNS, for example) so workloads in the VPC can resolve on-prem hostnames.
- **Response policy zones (RPZs)** — DNS-level allow-listing/deny-listing. Block resolution of known-malicious domains for any workload in the VPC.
- **Routing policies** — geo, weighted, failover. The Cloud DNS equivalent of Route 53 routing policies.

## Service Directory

**Service Directory** is a managed service registry for service discovery — a central place to register `service-name → endpoints` mappings, queryable via DNS, HTTP, or gRPC.

Most teams won't reach for it directly: Cloud Run services have stable URLs, GKE services have ClusterIPs + headless services, Cloud SQL connections go through PSC. Service Directory is the right answer when:

- You need *one* discovery surface across heterogeneous workloads (Cloud Run + GKE + on-prem + third-party).
- You want DNS-based discovery without writing Cloud DNS zones by hand.

It's quietly important infrastructure — the substrate behind Cloud DNS service zones and some PSC integrations — but rarely the headline of an architecture diagram.

## What carries into later chapters

The Global External Application LB is the front door to almost every public service in this course. Cloud Run gets it (notebook 04 → serverless NEGs). GKE gets it (notebook 04 → GKE Ingress or Gateway API). MIGs get it (notebook 03 → instance group backends). Cloud Armor and Cloud CDN attach to any of those without changing the backend.

Three habits to carry forward:

- **Global External Application LB by default for public HTTPS.** Regional variants exist for ops simplicity; the global one gets you anycast, GFE, Cloud CDN, and Cloud Armor in one package.
- **Cloud Armor in detection mode first, enforcement later.** Turning OWASP rules on cold is the fastest way to break legitimate traffic. Run in preview, watch the logs, then enforce.
- **Cloud DNS private zones over editing `/etc/hosts` files.** They cost nearly nothing and make every internal-name change a single API call instead of a fleet-wide reconfig.

Notebook 08 leaves networking and starts on storage of structured data — Cloud SQL, AlloyDB, and the cache layer Memorystore.